[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [GeoPandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)

# Spatial Predicates &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import matplotlib.pyplot as plt

COUNTRIES_URL = ("https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/data/nordic-countries.geojson")
local_copy = Path("../../data/nordic-countries.geojson")
countries = gpd.read_file(local_copy if local_copy.exists() else COUNTRIES_URL)


weather_stations = pd.DataFrame({
    "name": ["Bergen", "Oslo", "Svalbard", "Tromso", "Kirkenes"],
    "latitude": [60.39, 59.91, 78.22, 69.65, 69.73],
    "longitude": [5.32, 10.75, 15.65, 18.96, 30.05],
})


def mainland_norway():
    """Norway's outline south of 74 degrees north, without Svalbard, in UTM zone 33N."""
    parts = countries[countries["ADMIN"] == "Norway"].to_crs(25833).explode()
    return parts[parts.geometry.bounds["maxy"] < 8_200_000].union_all()


def station_network(n=300, seed=2026):
    """A made-up register of weather stations on mainland Norway, one row each."""
    rng = np.random.default_rng(seed)
    land = mainland_norway()
    x_min, y_min, x_max, y_max = land.bounds
    east, north = [], []
    while len(east) < n:
        x, y = rng.uniform(x_min, x_max), rng.uniform(y_min, y_max)
        if land.contains(shapely.Point(x, y)):
            east.append(x)
            north.append(y)
    lonlat = gpd.GeoSeries(gpd.points_from_xy(east, north), crs=25833).to_crs(4326)
    return pd.DataFrame({
        "station": [f"NS{i:03d}" for i in range(1, n + 1)],
        "latitude": lonlat.y.round(4).to_numpy(), "longitude": lonlat.x.round(4).to_numpy(),
        "elevation_m": np.clip(rng.gamma(1.6, 220, n), 2, 1900).round(0).astype(int),
        "owner": rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2]),
        "installed": rng.integers(1950, 2025, n),
    })


DISTRICT_CENTERS = {"South": (8.0, 58.6), "East": (10.8, 60.5), "West": (6.0, 60.8), "Mid": (11.0, 63.6),
                    "North": (14.5, 67.0), "Far North": (19.0, 69.4), "Northeast": (25.5, 70.0),
                    "Svalbard": (16.0, 78.5)}


def forecast_districts():
    """Eight made-up forecast districts that cover Norway, in UTM zone 33N."""
    norway = countries[countries["ADMIN"] == "Norway"].to_crs(25833).geometry.iloc[0]
    centers = gpd.GeoSeries(gpd.points_from_xy(*zip(*DISTRICT_CENTERS.values())), crs=4326).to_crs(25833)
    cells = shapely.voronoi_polygons(shapely.MultiPoint(list(centers)),
                                     extend_to=shapely.box(*norway.buffer(200_000).bounds)).geoms
    names = [next(name for name, center in zip(DISTRICT_CENTERS, centers) if cell.contains(center))
             for cell in cells]
    districts = gpd.GeoDataFrame({"district": names}, geometry=[cell.intersection(norway) for cell in cells],
                                 crs=25833)
    return districts.set_index("district").loc[list(DISTRICT_CENTERS)].reset_index()


network = station_network()
stations = gpd.GeoDataFrame(
    network, geometry=gpd.points_from_xy(network["longitude"], network["latitude"]), crs=4326
)


named = gpd.GeoDataFrame(
    weather_stations,
    geometry=gpd.points_from_xy(weather_stations["longitude"], weather_stations["latitude"]),
    crs=4326,
)


districts = forecast_districts()
stations_m = stations.to_crs(districts.crs)
print("geopandas", gpd.__version__, "| shapely", shapely.__version__, "|", len(stations), "made-up stations,",
      len(districts), "made-up districts")

geopandas 1.2.0 | shapely 2.2.0 | 300 made-up stations, 8 made-up districts


Each answer runs on its own after Setup. The stations and districts are made up, and the outlines are Natural Earth's.

**1.** Count the stations within each district with `within`, and print the counts in the districts' order.

In [2]:
counts = {name: stations_m.within(shape).sum()
          for name, shape in zip(districts["district"], districts.geometry)}
pd.Series(counts, name="stations")

South        32
East         56
West         43
Mid          69
North        27
Far North    25
Northeast    48
Svalbard      0
Name: stations, dtype: int64

A dictionary keeps the order its keys were added in, which is the districts' own order.

**2.** Print the districts that touch the North district.

In [3]:
shapes = districts.set_index("district").geometry
print(list(shapes[shapes.touches(shapes["North"])].index))

['Mid', 'Far North']


`touches` against North's own shape is `False` for North itself, since its interior overlaps itself, so only its neighbors are left.

**3.** Count the stations within 100 km of Bergen with `dwithin`.

In [4]:
bergen = named.to_crs(districts.crs).geometry.iloc[0]
print(stations_m.dwithin(bergen, 100_000).sum(), "stations within 100 km of Bergen")

13 stations within 100 km of Bergen


The distance is in the CRS's units, so 100 km is 100,000 meters.

**4.** Use `districts.sindex.query` with `predicate="intersects"` to find which district each of the five named stations falls in, and print the station and district names.

In [5]:
named_m = named.to_crs(districts.crs)
station_rows, district_rows = districts.sindex.query(named_m.geometry, predicate="intersects")
for s, d in zip(station_rows, district_rows):
    print(named_m["name"].iloc[s], "->", districts["district"].iloc[d])

Bergen -> West
Oslo -> East
Svalbard -> Svalbard


Tromso and Kirkenes are missing, because the 1:50m outline puts both just outside Norway, as the **Why GeoPandas** notebook found, and the districts were cut to that outline.

**5.** Find the stations that fall in more than one district's bounding box, and print how many of them each district holds by `within`.

In [6]:
candidates = districts.sindex.query(stations_m.geometry, sort=True)
per_station = np.bincount(candidates[0], minlength=len(stations_m))
several = stations_m[per_station > 1]
print(len(several), "stations in more than one box")
pd.Series({name: several.within(shape).sum()
           for name, shape in zip(districts["district"], districts.geometry)}, name="stations")

68 stations in more than one box


South        14
East         12
West         11
Mid          14
North         2
Far North     6
Northeast     9
Svalbard      0
Name: stations, dtype: int64

Each of these stations is in more than one box but in exactly one district, so the counts add up to the number of stations.

---

&#8592; **Back to:** [Spatial Predicates](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/07-spatial-predicates.ipynb)  &nbsp;&middot;&nbsp;  [GeoPandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)
